# GPT-3 복습과제 — In-context Learning 이해하기

**논문:** Brown et al., *Language Models are Few-Shot Learners* (NeurIPS 2020)

이 과제에서는 GPT-3 175B를 재현하지 않습니다. 대신 작은 공개 causal language model을 이용해 **zero-shot / one-shot / few-shot prompt의 차이**를 확인하고, 논문의 주요 결과와 benchmark contamination 문제를 간단히 복습합니다.

| Part | 내용 | 논문 연결 |
|---|---|---|
| 1 | Zero / One / Few-shot prompt 구성 | §2, Figure 2.1 |
| 2 | 작은 LM으로 in-context example의 영향 관찰 | §1–3 |
| 3 | 논문 결과 수치 해석 | §3 |
| 4 | Benchmark contamination 간단 실습 | §4 |
| 5 | 서술형 복습 | §5–8 |

> 목표는 코드를 어렵게 구현하는 것이 아니라, **무엇이 바뀌고 무엇은 학습되지 않는지** 이해하는 것입니다.

In [1]:
# 0. 환경 설정
!pip -q install transformers

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


---
## Part 1. Zero-shot / One-shot / Few-shot prompt 만들기

GPT-3 논문에서 세 설정의 차이는 **inference 시 제공하는 demonstration의 개수**입니다. 세 경우 모두 downstream task를 위해 model weight를 업데이트하지 않습니다.

아래에서는 간단한 영어→프랑스어 번역 task를 예로 들어 prompt를 만들어 봅니다.

### 문제 1-1. `build_prompt`의 빈칸을 채우세요

- `k=0`: instruction과 query만 사용
- `k=1`: demonstration 1개 사용
- `k>1`: 여러 demonstration 사용

In [3]:
examples = [
    ("sea otter", "loutre de mer"),
    ("peppermint", "menthe poivree"),
    ("plush giraffe", "girafe peluche"),
]

def build_prompt(examples, query, k=0):
    prompt = "Translate English to French.\n"
    for src, tgt in examples[:k]:                  # TODO ① 앞에서 k개의 demonstration 사용
        prompt += f"English: {src}\nFrench: {tgt}\n"
    prompt += f"English: {query}\nFrench:"
    return prompt

for k in [0, 1, 3]:
    print(f"\n--- k={k} ---")
    print(build_prompt(examples, "cheese", k))


--- k=0 ---
Translate English to French.
English: cheese
French:

--- k=1 ---
Translate English to French.
English: sea otter
French: loutre de mer
English: cheese
French:

--- k=3 ---
Translate English to French.
English: sea otter
French: loutre de mer
English: peppermint
French: menthe poivree
English: plush giraffe
French: girafe peluche
English: cheese
French:


In [4]:
# ✅ 확인 셀
assert "sea otter" not in build_prompt(examples, "cheese", 0)
assert "sea otter" in build_prompt(examples, "cheese", 1)
assert "plush giraffe" in build_prompt(examples, "cheese", 3)
print("✅ prompt 구성 확인")

✅ prompt 구성 확인


### 문제 1-2 (서술형)
Zero-shot → One-shot → Few-shot으로 갈 때 **변하는 것**과 **변하지 않는 것**을 각각 쓰세요. 특히 model parameter의 update 여부를 포함해 설명하세요.

**답:**
변하는 것: prompt에 넣는 demonstration 개수(0 → 1 → K개)
변하지 않는 것: model parameter

---
## Part 2. 작은 Language Model로 In-context Learning 관찰하기

실제 GPT-3는 공개 weight가 없고 175B 모델을 수업 과제로 재현하기도 어렵습니다. 여기서는 가벼운 `distilgpt2`를 사용해 **prompt에 example을 추가했을 때 출력이 어떻게 달라지는지**만 관찰합니다.

> 작은 모델의 결과가 GPT-3 논문의 성능을 재현하는 것은 아닙니다. 출력이 좋지 않아도 정상입니다.

In [5]:
MODEL_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
tokenizer.pad_token = tokenizer.eos_token

@torch.no_grad()
def generate(prompt, max_new_tokens=12):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("model loaded:", MODEL_ID)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model loaded: distilgpt2


### 문제 2-1. 같은 query를 zero / one / few-shot으로 비교하세요
`k` 값만 바꾸고 model weight는 전혀 학습하지 않습니다.

In [8]:
query = "cheese"

for k in [0, 1, 3]:
    prompt = build_prompt(examples, query, k=k)
    output = generate(prompt, max_new_tokens=12)
    print(f"\n--- k={k} ---")
    print(output)


--- k=0 ---
Translate English to French.
English: cheese
French: cheese
French: cheese
French: cheese
French:

--- k=1 ---
Translate English to French.
English: sea otter
French: loutre de mer
English: cheese
French: loutre de mer
English: cheese
French:

--- k=3 ---
Translate English to French.
English: sea otter
French: loutre de mer
English: peppermint
French: menthe poivree
English: plush giraffe
French: girafe peluche
English: cheese
French: coterie
French: coterie
French:


### 문제 2-2 (서술형)
위 실험에서 few-shot 결과가 zero-shot보다 좋아졌다면, 모델은 **어디에서 task 정보를 얻은 것**인가요? 반대로 결과가 좋아지지 않았더라도 이 실험이 GPT-3 논문의 주장을 반박한다고 할 수 없는 이유를 쓰세요.

**답:** few-shot이 좋아졌다면 task 정보는 가중치가 아니라 prompt 속 demonstration에서 얻은 것이다. 모델이 예시의 입력-출력 형식과 패턴을 보고 무슨 task인지 추론하는 것이라고 볼 수 있다.
실험 결과를 보면 few-shot이 좋아졌다고 보기 어렵다.
k=0, 1에서는 이전 출력을 그대로 반복했다.
k=3에서는 coterie처럼 프랑스어 느낌의 단어를 냈지만 틀렸다.
즉 “French: ...” 같은 형식은 따라 하지만 번역 능력 자체는 없는 상태이다.
그래도 논문을 반박하지 못하는 이유는 다음과 같다.
논문의 핵심 주장은 in-context learning 능력이 모델 규모에 따라 커진다는 것이다. 작은 모델은 K를 늘려도 개선이 작고 175B에서 크게 나타난다.
distilgpt2는 약 8천만 파라미터 수준이고 영어 중심이라 GPT-3 175B와 비교 대상이 안 된다.

---
## Part 3. 논문의 주요 결과 읽기

아래는 논문에서 보고한 GPT-3 175B의 일부 결과입니다. 숫자를 직접 계산해 보면서 결과를 해석합니다.

In [10]:
results = {
    "LAMBADA": {"zero": 76.2, "one": 72.5, "few": 86.4, "previous_sota": 68.0},
    "TriviaQA": {"zero": 64.3, "one": 68.0, "few": 71.2, "previous_sota": 68.0},
    "CoQA": {"zero": 81.5, "one": 84.0, "few": 85.0, "previous_sota": 90.7},
}

for task, score in results.items():
    gain = score["few"] - score["zero"]                            # TODO ③ few-shot - zero-shot
    print(f"{task:10s} | zero→few gain: {gain:+.1f} points")

LAMBADA    | zero→few gain: +10.2 points
TriviaQA   | zero→few gain: +6.9 points
CoQA       | zero→few gain: +3.5 points


In [11]:
# ✅ 확인 셀
assert round(results["LAMBADA"]["few"] - results["LAMBADA"]["zero"], 1) == 10.2
assert round(results["TriviaQA"]["few"] - results["TriviaQA"]["zero"], 1) == 6.9
print("✅ 계산 확인")

✅ 계산 확인


### 문제 3-1 (서술형)
1. 세 task 모두에서 `few-shot > zero-shot`인가요? 예외가 있다면 무엇인가요?
2. LAMBADA와 TriviaQA의 결과만 보고 “GPT-3가 모든 NLP task에서 fine-tuned SOTA를 능가했다”고 결론 내려도 될까요? 이유를 쓰세요.

**답:** 1. 세 task 모두 few-shot > zero-shot. LAMBADA 86.4 > 76.2, TriviaQA 71.2 > 64.3, CoQA 85.0 > 81.5. 다만 예외로 LAMBADA는 one-shot(72.5)이 zero-shot(76.2)보다 낮다. 논문은 zero-shot과 one/few-shot의 prompt 형식이 다르고 예시 하나로는 패턴을 인식하기 어려운 듯하다고 설명한다.
2. CoQA는 few-shot 85.0이 fine-tuned SOTA 90.7에 못 미친다.
논문 자체가 WiC(49.4%, 랜덤 수준), ANLI, RACE, QuAC 같은 약한 task를 인정한다.
LAMBADA와 TriviaQA 같은 일부 task만 근거로 일반화할 수 없고 LAMBADA는 오염 가능성도 있어서 해석에 주의가 필요하다.

---
## Part 4. Benchmark Contamination 간단 확인

GPT-3는 Common Crawl을 포함한 대규모 웹 데이터로 사전학습되었습니다. 따라서 benchmark의 test 문장이 웹에 공개되어 있었다면 training corpus에 포함될 수 있습니다.

여기서는 실제 GPT-3 training corpus 대신 작은 문자열을 사용해 **n-gram overlap**의 아이디어만 확인합니다.

### 문제 4-1. 두 텍스트 사이의 n-gram overlap을 계산하세요

In [13]:
def word_ngrams(text, n=4):
    words = re.findall(r"[a-z]+", text.lower())
    return {tuple(words[i:i+n]) for i in range(len(words) - n + 1)}

def overlap_ratio(train_text, test_text, n=4):
    train_ng = word_ngrams(train_text, n)
    test_ng = word_ngrams(test_text, n)
    overlap = train_ng & test_ng                         # TODO ④ 두 n-gram 집합의 교집합
    return len(overlap) / max(len(test_ng), 1)

train_text = "the quick brown fox jumps over the lazy dog and runs away"
test_clean = "language models can learn patterns from examples in context"
test_dirty = "the quick brown fox jumps over the fence"

print("clean overlap:", overlap_ratio(train_text, test_clean))
print("dirty overlap:", overlap_ratio(train_text, test_dirty))

clean overlap: 0.0
dirty overlap: 0.8


In [14]:
# ✅ 확인 셀
assert overlap_ratio(train_text, test_clean) == 0
assert overlap_ratio(train_text, test_dirty) > 0
print("✅ overlap 확인")

✅ overlap 확인


### 문제 4-2 (서술형)
Benchmark test example이 pretraining corpus에 포함되어 있다면 왜 성능을 해석하기 어려워지나요? **generalization**과 **memorization**이라는 단어를 사용해 설명하세요.

**답:** test 예시가 pretraining corpus에 있으면 모델이 문제를 풀어서 맞힌 건지, 학습 중 본 것을 외워서 맞힌 건지 구분할 수 없다. 그러면 벤치마크 점수가 모델의 진짜 일반화 능력보다 부풀려진다.
논문은 13-gram overlap으로 깨끗한 부분집합을 만들어 성능을 비교했다. 대부분 변화가 작았지만 PIQA와 Winograd는 asterisk를 붙였고 오염이 심한 LM 벤치마크는 아예 제외했다.
한계로는 필터링 버그 때문에 일부가 제거되지 않았고 clean subset이 원래 분포보다 더 쉽거나 어려울 수 있다는 점이 있다.

---
## Part 5. 서술형 복습

**5-1.** GPT-3 논문에서 말하는 *in-context learning*과 일반적인 fine-tuning의 가장 중요한 차이를 설명하세요.

**답:** 학습 결과가 가중치에 저장되느냐(FT) vs 입력 조건으로만 쓰이고 사라지느냐(ICL)

**5-2.** 논문에서 125M부터 175B까지 여러 크기의 모델을 함께 학습한 이유는 무엇인가요? 단순히 “가장 큰 모델의 성능을 보기 위해서”보다 구체적으로 쓰세요.

**답:** 같은 구조, 데이터, 학습 토큰으로 크기만 바꿔서 성능이 규모에 따라 어떻게 변하는지 체계적으로 보려는 것이다. loss가 compute에 대해 power-law로 매끄럽게 감소하는지, zero/one/few-shot 격차가 클수록 큰 모델이 더 좋은 meta-learner인지, 산술 같은 능력이 13B → 175B에서 급등하는지(Fig 3.10) 확인하고자 한 것이다.

**5-3.** GPT-3는 일부 task에서는 강한 few-shot 성능을 보였지만 NLI, 일부 reading comprehension 등에서는 약했습니다. 이 결과가 논문의 주장에 어떤 제한을 주는지 설명하세요.

**답:** 특히 두 문장을 비교하는 유형에서 약하다는 패턴이 있다. 저자들은 단방향 구조 탓일 수 있다고 추측하지만 검증하지 못했다.
또 few-shot이 새로운 task를 배우는 건지 사전학습에서 본 task를 인식하는 건지 불분명하다고 스스로 인정했다.

**5-4.** GPT-3가 생성한 뉴스 기사를 사람이 약 52% 정확도로 판별했다는 결과는 어떤 가능성과 위험을 동시에 보여주나요?

**답:** 생성 품질이 사람이 구분하기 어려울 정도로 높아서 글쓰기 보조, 초안 작성 등에 유용할 수 있다. 하지만 허위정보, 스팸, 피싱 같은 오용의 진입 장벽이 낮아지고 사람의 눈으로는 걸러내기 어려워진다.

---
## 참고 자료
- Brown et al. (2020), *Language Models are Few-Shot Learners*: Approach (§2), Results (§3), Benchmark contamination (§4), Limitations (§5), Broader Impacts (§6)
- OpenAI GPT-3 repository: 논문의 synthetic task data와 benchmark overlap 관련 자료가 공개되어 있습니다.

> 이 과제는 GPT-3 175B의 원 실험을 재현하는 과제가 아니라, 논문의 핵심 메커니즘과 결과 해석을 작은 예제로 복습하기 위한 과제입니다.